# 🚀 Astro Rasayana: Multi-Agent Space Medicine System with Gemini, Live Evidence Triangulation & Structured Output

## 🔍 Project Overview

Synthetic pharmaceuticals degrade rapidly during long-duration spaceflight under cosmic radiation, and crews cannot resupply mid-mission. This notebook implements **Astro Rasayana**, a multi-agent system that screens classical Ayurvedic Rasayana herbs as candidate self-sustaining biological countermeasures for spaceflight stress, and critically refuses to recommend anything it cannot back up with live, independently sourced evidence.

Three specialist agents are orchestrated by a **Supervisor Agent**:

1.  **🌿 The Botanist Agent** - extracts classical Rasayana herbs and their active compounds from a structured Himalayan flora dataset.
2.  **🧬 The Clinician Agent** - queries the live **PubMed API** to check whether a compound has literature support against the target physiological stressor.
3.  **🛰️ The Flight Surgeon Agent** - queries the live **NASA GeneLab API** for spaceflight datasets relevant to that same stressor.

The Supervisor enforces the **Triangulation Mandate**: a herb is only accepted if it clears all three legs — a Rasayana classification, a verified PubMed result, and a NASA GeneLab dataset on the same target. That gate is enforced **in code, from a deterministic log of tool calls**, not from whatever the language model's closing message happens to say.

**🤖 Key Generative AI Capabilities Demonstrated:**

*   **Multi-Agent Orchestration:** A Supervisor Agent drives three specialist tool-using agents through Gemini's automatic function calling.
*   **Live Tool Use / Grounding:** Real-time calls to the PubMed E-utilities API and the NASA GeneLab API - not cached or hallucinated citations.
*   **Deterministic Governance:** The accept/reject decision for every herb is rebuilt in plain Python from the logged tool calls, so the LLM's prose can never overstate the evidence.
*   **Structured Output:** Findings are assembled into a typed report dictionary, ready for either the visual display function or a downstream JSON/Markdown export.
*   **Agentic System Design:** An explicit system instruction forces exhaustive, per-herb tool coverage rather than early stopping.

## ⚙️ Step 1: Setting Up the Environment

This first code cell installs and imports what the rest of the notebook needs: `pandas` for the Himalayan flora dataset, `requests` for the live PubMed and NASA GeneLab calls, the `google-genai` SDK for Gemini's automatic function calling, `ipywidgets` for the interactive interface built in Step 10, and standard library helpers (`json`, `inspect`, `functools`, `datetime`) used by the governance layer in Step 7. It also lists any files Kaggle has mounted under `/kaggle/input`, for visibility into attached datasets.

In [1]:
# Step 1: Setting Up the Environment
import warnings
warnings.filterwarnings("ignore")

!pip install -q google-genai ipywidgets

import os
import re
import json
import inspect
import functools
from collections import defaultdict
from datetime import datetime, timezone

import pandas as pd
import requests

print("Environment ready.")

# Visibility into any Kaggle-attached datasets
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            print(os.path.join(root, f))

Environment ready.
/kaggle/input/competitions/vibecoding-agents-capstone-project/NOTE.md


## 🔑 Step 2: Gemini API Configuration and Model Selection

This cell configures the connection to Google's Gemini API. It retrieves the API key from **Kaggle Secrets** (`Add-ons > Secrets`) when running on Kaggle, or falls back to the `GEMINI_API_KEY` environment variable for local runs. `gemini-2.5-flash` is selected as the orchestration model — it is fast and reliable at automatic function calling, which is what the Supervisor Agent depends on in Step 8.

In [2]:
# Step 2: Gemini API Configuration and Model Selection
try:
    from kaggle_secrets import UserSecretsClient
    GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Add it via Kaggle Secrets (Add-ons > Secrets), "
        "or set it as an environment variable before running this notebook."
    )

from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-2.5-flash"
print(f"Gemini client configured. Model: {GEMINI_MODEL}")

Gemini client configured. Model: gemini-2.5-flash


## 📚 Step 3: Loading the Himalayan Flora Dataset

The Botanist Agent needs a CSV with `scientific_name`, `traditional_property`, `active_compounds`, and `clinical_targets` columns. This cell looks for it at the project's default path (`data/himalayan_flora.csv`) or a Kaggle-mounted `+ Add Data` path.

**Fallback:** if neither is found, the cell writes a small illustrative sample of five well-known Himalayan Rasayana herbs to `data/himalayan_flora.csv`, so the rest of the notebook still runs end-to-end for demonstration. Attach the full dataset via Kaggle's `+ Add Data` for the real pipeline.

In [3]:
# Step 3: Loading the Himalayan Flora Dataset
CSV_CANDIDATES = [
    "data/himalayan_flora.csv",
    "/kaggle/input/himalayan-flora/himalayan_flora.csv",
]

def _sample_himalayan_flora() -> pd.DataFrame:
    """Small illustrative fallback so the notebook runs end-to-end without the full dataset."""
    return pd.DataFrame([
        {
            "scientific_name": "Withania somnifera",
            "traditional_property": "Rasayana",
            "active_compounds": "withanolides",
            "clinical_targets": "oxidative stress, immune dysregulation, muscle atrophy",
        },
        {
            "scientific_name": "Rhodiola imbricata",
            "traditional_property": "Rasayana",
            "active_compounds": "salidroside",
            "clinical_targets": "oxidative stress, radiation-induced DNA damage",
        },
        {
            "scientific_name": "Hippophae rhamnoides",
            "traditional_property": "Rasayana",
            "active_compounds": "flavonoids, carotenoids",
            "clinical_targets": "oxidative stress, immune dysregulation",
        },
        {
            "scientific_name": "Podophyllum hexandrum",
            "traditional_property": "Rasayana",
            "active_compounds": "podophyllotoxin",
            "clinical_targets": "radiation-induced DNA damage",
        },
        {
            "scientific_name": "Bergenia ligulata",
            "traditional_property": "Rasayana",
            "active_compounds": "bergenin",
            "clinical_targets": "bone density loss",
        },
    ])

CSV_PATH = next((p for p in CSV_CANDIDATES if os.path.exists(p)), None)
if CSV_PATH is None:
    os.makedirs("data", exist_ok=True)
    CSV_PATH = "data/himalayan_flora.csv"
    sample_df = _sample_himalayan_flora()
    sample_df.to_csv(CSV_PATH, index=False)
    print(f"[Fallback] No Himalayan flora dataset found on disk. Wrote a {len(sample_df)}-row "
          f"sample to {CSV_PATH} so the notebook can run end-to-end. Attach the full dataset via "
          f"'+ Add Data' for the real pipeline.")
else:
    print(f"Using Himalayan flora dataset at: {CSV_PATH}")

pd.read_csv(CSV_PATH).head()

[Fallback] No Himalayan flora dataset found on disk. Wrote a 5-row sample to data/himalayan_flora.csv so the notebook can run end-to-end. Attach the full dataset via '+ Add Data' for the real pipeline.


,scientific_name,traditional_property,active_compounds,clinical_targets
0,Withania somnifera,Rasayana,withanolides,"oxidative stress, immune dysregulation, muscle..."
1,Rhodiola imbricata,Rasayana,salidroside,"oxidative stress, radiation-induced DNA damage"
2,Hippophae rhamnoides,Rasayana,"flavonoids, carotenoids","oxidative stress, immune dysregulation"
3,Podophyllum hexandrum,Rasayana,podophyllotoxin,radiation-induced DNA damage
4,Bergenia ligulata,Rasayana,bergenin,bone density loss


## 🌿 Step 4: The Botanist Agent — Rasayana Herb Filtering

`filter_rasayana_herbs` reads the flora CSV and keeps only rows classified as `Rasayana`. If a `target_stressor` is given, it pre-filters to herbs whose `clinical_targets` field looks related — by exact phrase match first, then by keyword overlap to tolerate differing terminology (e.g. *"radiation-induced DNA damage"* vs. *"DNA repair"*). If nothing matches, it deliberately **falls back to returning every Rasayana herb** rather than stopping early — this CSV field is only a coarse pre-filter, and the real relevance check happens downstream against live PubMed and NASA GeneLab evidence in Steps 5–6.

In the full `astro_rasayana` repository this function is exposed over MCP by `mcp_servers/botanist.py`. Here it is called directly as a Gemini tool function — exactly what `supervisor.py` already does under the hood, since the `@mcp.tool()` decorator does not change how the plain function behaves.

In [4]:
# Step 4: The Botanist Agent
_STOPWORDS = {
    "induced", "related", "associated", "and", "or", "the", "a", "of", "in",
    "on", "to", "for", "due", "from", "with",
}


def _keywords(phrase: str) -> list[str]:
    return [w for w in re.split(r"[\s\-]+", phrase.lower()) if w and w not in _STOPWORDS]


def _row_matches(clinical_targets: str, target_stressor: str) -> bool:
    if not isinstance(clinical_targets, str):
        return False
    text = clinical_targets.lower()
    phrase = target_stressor.lower()

    # 1. Exact phrase match (best case, keep it fast when it works).
    if phrase in text:
        return True

    # 2. Keyword overlap match -- handles differing terminology between
    #    the preset/custom stressor label and however the CSV author
    #    phrased the clinical_targets column (e.g. "radiation-induced DNA
    #    damage" vs. "DNA repair" or "Radiation protection").
    kws = _keywords(phrase)
    return any(k in text for k in kws)


def filter_rasayana_herbs(csv_path: str, target_stressor: str = "") -> list[dict]:
    """Reads a CSV of Himalayan flora and returns herbs classified as Rasayana.

    If target_stressor is provided, herbs are preferentially filtered to
    those whose clinical_targets field is related to it (exact phrase or
    keyword overlap, case-insensitive). If nothing matches -- which usually
    means the CSV uses different terminology than the requested stressor --
    this falls back to returning all Rasayana herbs, since this CSV field
    is only a coarse pre-filter. The actual relevance check happens
    downstream via live PubMed and NASA GeneLab evidence, which is the
    real Triangulation Mandate gate.
    """
    try:
        df = pd.read_csv(csv_path)
        rasayana_df = df[df['traditional_property'].str.contains('Rasayana', na=False, case=False)]
        columns = ['scientific_name', 'active_compounds', 'clinical_targets']

        if not target_stressor:
            return rasayana_df[columns].to_dict(orient='records')

        mask = rasayana_df['clinical_targets'].apply(lambda t: _row_matches(t, target_stressor))
        filtered_df = rasayana_df[mask]

        if filtered_df.empty:
            print(
                f"[Botanist] No CSV rows matched target '{target_stressor}' by keyword. "
                f"Returning all {len(rasayana_df)} Rasayana herbs for evidence-based triangulation "
                f"instead of stopping early."
            )
            return rasayana_df[columns].to_dict(orient='records')

        return filtered_df[columns].to_dict(orient='records')
    except Exception as e:
        return [{"error": str(e)}]


print("Botanist Agent ready: filter_rasayana_herbs()")

Botanist Agent ready: filter_rasayana_herbs()


## 🧬 Step 5: The Clinician Agent — Live PubMed Validation

`search_pubmed_clinical_trials` queries the **live NCBI PubMed E-utilities API** (`esearch` + `esummary`) to check whether a herb's active compound has literature linking it to the target clinical stressor. It first tries a strict query restricted to `Clinical Trial[Publication Type]`. If that returns nothing, it falls back to a broader search across all publication types and labels the result `preclinical_or_review` instead of `clinical_trial` — so the final report never overstates how strong the evidence actually is.

`herb_name` is passed through only for traceability, so the Supervisor can attribute PubMed hits back to the right herb; it does not affect the search query itself. This mirrors `mcp_servers/clinician.py` in the full repository, called here directly rather than over MCP transport.

In [5]:
# Step 5: The Clinician Agent
ESEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
ESUMMARY_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"


def _pubmed_search(query: str, retmax: int = 3) -> list[str]:
    params = {"db": "pubmed", "term": query, "retmode": "json", "retmax": retmax}
    response = requests.get(ESEARCH_URL, params=params, timeout=10)
    response.raise_for_status()
    return response.json().get("esearchresult", {}).get("idlist", [])


def _pubmed_titles(pmids: list[str]) -> dict[str, str]:
    if not pmids:
        return {}
    params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "json"}
    response = requests.get(ESUMMARY_URL, params=params, timeout=10)
    response.raise_for_status()
    result = response.json().get("result", {})
    return {pmid: result.get(pmid, {}).get("title", "Untitled") for pmid in pmids}


def search_pubmed_clinical_trials(herb_name: str, active_compound: str, clinical_target: str) -> list[dict]:
    """Queries PubMed for evidence linking a compound to a clinical target.

    herb_name is used only for traceability so the Supervisor can attribute
    results back to the correct herb regardless of how the compound name is
    phrased in the query itself. It is not used to build the search query.

    The search first tries a strict query restricted to Clinical Trial
    publication types. If nothing is found, it falls back to a broader
    search and labels the evidence_tier as 'preclinical_or_review' rather
    than 'clinical_trial', so downstream reporting never overstates the
    strength of the evidence.
    """
    try:
        strict_query = (
            f"{active_compound}[Title/Abstract] AND {clinical_target}[Title/Abstract] "
            f"AND Clinical Trial[Publication Type]"
        )
        pmids = _pubmed_search(strict_query)
        evidence_tier = "clinical_trial"

        if not pmids:
            broad_query = f"{active_compound}[Title/Abstract] AND {clinical_target}[Title/Abstract]"
            pmids = _pubmed_search(broad_query)
            evidence_tier = "preclinical_or_review"

        if not pmids:
            return [{"herb_name": herb_name, "status": "No PubMed evidence found.", "evidence_tier": "none"}]

        titles = _pubmed_titles(pmids)
        return [
            {
                "herb_name": herb_name,
                "pubmed_id": pmid,
                "title": titles.get(pmid, "Untitled"),
                "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
                "evidence_tier": evidence_tier,
            }
            for pmid in pmids
        ]

    except Exception as e:
        return [{"herb_name": herb_name, "error": str(e)}]


print("Clinician Agent ready: search_pubmed_clinical_trials()")

Clinician Agent ready: search_pubmed_clinical_trials()


## 🛰️ Step 6: The Flight Surgeon Agent — NASA GeneLab Validation

`search_genelab_biomarkers` queries the **live NASA GeneLab API** for spaceflight datasets related to the target physiological stressor. NASA GeneLab studies model organisms and astronaut biology in spaceflight, not individual phytochemicals, so a compound-specific hit is rare. The tool first tries a compound-specific search (stressor + active compound + "microgravity"); if that finds nothing, it falls back to a stressor-only search and labels the result `target_level` instead of `compound_specific`.

This distinction matters for how the final report should be read: a `target_level` match confirms the physiological target is genuinely perturbed by spaceflight — **not** that the herb itself has been tested in space. This mirrors `mcp_servers/flight_surgeon.py` in the full repository.

In [6]:
# Step 6: The Flight Surgeon Agent
GENELAB_SEARCH_URL = "https://genelab-data.ndc.nasa.gov/genelab/data/search"


def _genelab_search(term: str, size: int = 3) -> list[dict]:
    params = {"term": term, "type": "cgene", "size": size}
    response = requests.get(GENELAB_SEARCH_URL, params=params, timeout=10)
    response.raise_for_status()
    hits = response.json().get("hits", {}).get("hits", [])
    results = []
    for hit in hits:
        source = hit.get("_source", {})
        accession = source.get("Accession", "Unknown")
        results.append({
            "accession": accession,
            "title": source.get("Project_Title", "Untitled"),
            "url": f"https://osdr.nasa.gov/bio/repo/data/studies/{accession}",
        })
    return results


def search_genelab_biomarkers(stressor: str, herb_name: str = "", active_compound: str = "") -> list[dict]:
    """Queries NASA GeneLab for spaceflight datasets related to a physiological stressor.

    herb_name is used only for traceability so the Supervisor can attribute
    results back to the correct herb.

    NASA GeneLab studies model organisms and astronaut biology in spaceflight,
    not individual phytochemicals, so a compound-specific hit is rare. This
    tool first attempts a compound-specific search when active_compound is
    given. If nothing is found, it falls back to a stressor-level search and
    labels the result 'target_level' rather than 'compound_specific' -- this
    keeps the report honest about what NASA's data actually validates: that
    the physiological target is genuinely perturbed by spaceflight, not that
    the herb itself has been tested in space.
    """
    try:
        if active_compound:
            results = _genelab_search(f"{stressor} {active_compound} microgravity")
            if results:
                for r in results:
                    r["herb_name"] = herb_name
                    r["evidence_tier"] = "compound_specific"
                return results

        results = _genelab_search(f"{stressor} microgravity")
        if not results:
            return [{
                "herb_name": herb_name,
                "status": f"No GeneLab datasets found for {stressor}.",
                "evidence_tier": "none",
            }]

        for r in results:
            r["herb_name"] = herb_name
            r["evidence_tier"] = "target_level"
        return results

    except Exception as e:
        return [{"herb_name": herb_name, "error": str(e)}]


print("Flight Surgeon Agent ready: search_genelab_biomarkers()")

Flight Surgeon Agent ready: search_genelab_biomarkers()


## ⚖️ Step 7: The Governance Layer — Tool Call Logging & the Triangulation Mandate

This is the part that keeps the system honest, and it deliberately contains **no LLM calls at all**.

*   **`make_logged`** wraps each of the three agent functions above. Every call it intercepts is recorded with its normalized arguments and raw result into `call_log` — independent of whatever the model's final chat message claims it did.
*   **`build_report`** cross-references `call_log` against the herb list from the Botanist Agent. For each herb it checks: was the Clinician Agent queried, and did it return a real PubMed ID? Was the Flight Surgeon Agent queried, and did it return a real GeneLab accession? A herb is marked `PASS` **only if both are true**; otherwise it is `REJECTED` with an explicit, itemized reason (e.g. *"Flight Surgeon Agent was never queried for this herb"*).

Because the report is assembled deterministically from the log rather than parsed out of the model's prose, the Triangulation Mandate is enforced **in code**, not just requested in a system prompt.

In [7]:
# Step 7: Tool Call Logging and the Deterministic Report Builder

def make_logged(func, call_log: list):
    """Wraps an agent function so every call is captured with args + raw result,
    independent of whatever the LLM's final chat message claims it did."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        bound = inspect.signature(func).bind(*args, **kwargs)
        bound.apply_defaults()
        call_log.append({
            "tool": func.__name__,
            "args": dict(bound.arguments),
            "result": result,
        })
        return result
    return wrapper


def build_report(csv_path, stressor, mission_duration, herb_list, call_log):
    """The actual governance gate. Rebuilds PASS/REJECTED status per herb from
    the logged tool calls -- this is the Triangulation Mandate enforced in code."""
    clinician_by_herb = defaultdict(list)
    genelab_by_herb = defaultdict(list)

    for call in call_log:
        herb_name = call["args"].get("herb_name", "")
        if call["tool"] == "search_pubmed_clinical_trials":
            clinician_by_herb[herb_name].extend(call["result"])
        elif call["tool"] == "search_genelab_biomarkers":
            genelab_by_herb[herb_name].extend(call["result"])

    findings = []
    evaluated_count = 0

    for herb in herb_list:
        name = herb.get("scientific_name", "Unknown")
        pubmed_results = clinician_by_herb.get(name, [])
        genelab_results = genelab_by_herb.get(name, [])

        valid_pubmed = [e for e in pubmed_results if "pubmed_id" in e]
        valid_genelab = [e for e in genelab_results if "accession" in e]

        if pubmed_results and genelab_results:
            evaluated_count += 1

        reasons = []
        if not pubmed_results:
            reasons.append("Clinician Agent was never queried for this herb")
        elif not valid_pubmed:
            reasons.append("No PubMed evidence found linking the compound to the target")
        if not genelab_results:
            reasons.append("Flight Surgeon Agent was never queried for this herb")
        elif not valid_genelab:
            reasons.append("No NASA GeneLab dataset found for the target")

        status = "PASS" if (valid_pubmed and valid_genelab) else "REJECTED"

        findings.append({
            "scientific_name": name,
            "active_compounds": herb.get("active_compounds", ""),
            "status": status,
            "reasons": reasons,
            "pubmed_evidence": valid_pubmed,
            "genelab_evidence": valid_genelab,
        })

    return {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_stressor": stressor,
        "mission_duration_months": mission_duration,
        "csv_source": csv_path,
        "herbs_identified": len(herb_list),
        "herbs_evaluated": evaluated_count,
        "herbs_passed": sum(1 for f in findings if f["status"] == "PASS"),
        "herbs_rejected": sum(1 for f in findings if f["status"] == "REJECTED"),
        "findings": findings,
        "raw_call_log": call_log,
        "limitations": [
            "Triangulation links a traditional Rasayana classification, PubMed literature, and "
            "NASA GeneLab data on the same physiological target. It is not evidence of a proven "
            "causal or clinical effect of the herb itself in spaceflight.",
            "NASA GeneLab predominantly studies model organisms and astronaut biology in "
            "spaceflight, not individual phytochemicals. A 'target_level' GeneLab match confirms "
            "the physiological target is genuinely perturbed by spaceflight, not that the herb "
            "has been tested in space.",
            "PubMed evidence tiered as 'preclinical_or_review' reflects non-clinical-trial "
            "literature (in-vitro, animal studies, or reviews) and should be weighted accordingly.",
            "This report is a research screening tool, not a clinical or regulatory "
            "recommendation. No herb listed here has regulatory approval or clinical validation "
            "for spaceflight use.",
        ],
    }


print("Governance layer ready: make_logged(), build_report()")

Governance layer ready: make_logged(), build_report()


## 🧑‍🚀 Step 8: Orchestrating the Multi-Agent Pipeline

`run_astro_pipeline` is the Supervisor Agent. It:

1.  Wraps all three agent functions with `make_logged` so every call lands in a shared `call_log`.
2.  Sends Gemini a strict system instruction: call the Botanist Agent once, then call **both** the Clinician Agent and the Flight Surgeon Agent for **every** herb the Botanist returns — no skipping, no early stopping.
3.  Lets Gemini's automatic function calling drive the three tools via `types.GenerateContentConfig(tools=[...])`.
4.  Ignores the model's closing chat message for the actual findings — it only confirms how many herbs were evaluated — and instead calls `build_report` on `call_log` to get the governed, code-verified result.

This is the same separation of concerns as the standalone `agents/supervisor.py` script in the repository: the LLM plans and calls tools, but a plain Python function decides what counts as evidence.

In [8]:
# Step 8: The Supervisor Agent

STRESSOR_PRESETS = [
    "oxidative stress",
    "bone density loss",
    "muscle atrophy",
    "immune dysregulation",
    "radiation-induced DNA damage",
]


def run_astro_pipeline(csv_path: str, stressor: str, mission_duration: int | None = None) -> dict:
    """Runs the full Botanist -> Clinician -> Flight Surgeon -> Supervisor pipeline
    and returns a governed report dict (see build_report for its shape)."""
    call_log = []
    botanist_tool = make_logged(filter_rasayana_herbs, call_log)
    clinician_tool = make_logged(search_pubmed_clinical_trials, call_log)
    flight_surgeon_tool = make_logged(search_genelab_biomarkers, call_log)

    system_instruction = (
        "You are the Astro Pharmacognosy Supervisor Agent. "
        f"Your mission is to identify Rasayana herbs that could serve as spaceflight "
        f"countermeasures for the target physiological stressor: '{stressor}'. "
        "You must enforce the Triangulation Mandate: a herb can only be recommended if there "
        "is (1) a Rasayana classification, (2) PubMed evidence linking its active compound to "
        "the target, and (3) a NASA GeneLab dataset relevant to the target. "
        "Follow these steps exactly: "
        "1. Call filter_rasayana_herbs once with the given csv_path and target_stressor. "
        "2. For EVERY herb returned, call search_pubmed_clinical_trials, passing herb_name set "
        "to the exact scientific_name from the botanist result, plus its active_compound and "
        "the clinical_target. "
        "3. For EVERY herb returned, also call search_genelab_biomarkers, passing the same "
        "herb_name, the target stressor, and the herb's active_compound. "
        "4. Do not skip any herb, and do not stop early. "
        "5. Your final reply only needs to confirm how many herbs you evaluated -- a separate, "
        "code-based process assembles the actual report from your tool call results, so do not "
        "restate findings in your closing message."
    )

    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        tools=[botanist_tool, clinician_tool, flight_surgeon_tool],
        temperature=0.2,
    )

    chat = client.chats.create(model=GEMINI_MODEL, config=config)
    response = chat.send_message(
        f"Begin the Astro Pharmacognosy analysis using {csv_path} and target stressor '{stressor}'."
    )

    herb_calls = [c for c in call_log if c["tool"] == "filter_rasayana_herbs"]
    if not herb_calls:
        return {"error": "The Botanist Agent was never called. No report can be generated."}

    herb_list = [h for h in herb_calls[0]["result"] if "scientific_name" in h]
    if not herb_list:
        return {"error": f"No Rasayana herbs matched target stressor '{stressor}' in the dataset."}

    report = build_report(csv_path, stressor, mission_duration, herb_list, call_log)
    report["agent_closing_note"] = response.text
    return report


print("Supervisor Agent ready: run_astro_pipeline()")

Supervisor Agent ready: run_astro_pipeline()


## 🎨 Step 9: A Visually Appealing Report Display Function

Instead of writing separate JSON and Markdown files and reading them back to see the result, `display_astro_report` renders the governed report dict directly and interactively, in one call — the same simplification `display_diagnosis` gave the ADA project.

*   **Input:** the dict returned by `run_astro_pipeline` (or `build_report`).
*   **Functionality:**
    *   Validates the input and surfaces any pipeline-level error clearly (e.g. the Botanist Agent never being called).
    *   Shows a **Completeness Check** row of stat cards: herbs identified, evaluated, passed, and rejected.
    *   Renders each **PASS** herb as a green card with its active compounds and the actual PubMed / NASA GeneLab evidence links and tiers.
    *   Renders **REJECTED** herbs as a compact list with the itemized reason for rejection.
    *   Always prints the **Limitations** section, so the report can never be read as a clinical or regulatory recommendation.
*   **Output:** rendered with `IPython.display.HTML`, directly in the notebook output area.

In [9]:
# Step 9: display_astro_report — the simplified output function

from IPython.display import display, HTML

def display_astro_report(report):
    """Renders a governed Astro Rasayana report as a visually structured HTML card,
    in one call -- replacing separate JSON + Markdown file writes with a single
    interactive display, the way display_diagnosis() does for ADA."""

    # --- Input validation ---
    if not isinstance(report, dict):
        display(HTML(f"""
        <div style="border: 2px solid orange; padding: 15px; background-color: #fff8e1; color: #6f4f00;">
            <h2><span style="color:orange;">🤔</span> Input Error</h2>
            <p><strong>Details:</strong> The provided report input is not a valid dictionary.</p>
            <pre style="white-space: pre-wrap;">Input Type: {type(report)}</pre>
        </div>
        """))
        return

    # --- Pipeline-level error handling ---
    if "error" in report and "findings" not in report:
        display(HTML(f"""
        <div style="border: 2px solid red; padding: 15px; background-color: #fff0f0; color: #a00;">
            <h2><span style="color:red;">⚠️</span> Pipeline Error</h2>
            <p><strong>Details:</strong> {report['error']}</p>
        </div>
        """))
        return

    current_date = datetime.now(timezone.utc).strftime("%B %d, %Y")

    html = f"""
    <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; max-width: 850px; margin: 10px auto; padding: 25px; border: 1px solid #ccc; border-radius: 10px; background-color: #0b1024; color: #e8eaf6; box-shadow: 0 5px 15px rgba(0,0,0,0.25);">
        <h2 style="border-bottom: 2px solid #5c6bc0; padding-bottom: 10px; text-align: center; margin-bottom: 10px;">🚀 Astro Pharmacognosy Triangulation Report 🌿</h2>
        <p style="text-align:center; color:#aab4e0; margin-top:0;">Target stressor: <strong>{report.get('target_stressor', 'N/A')}</strong>
    """
    if report.get("mission_duration_months"):
        html += f" &nbsp;|&nbsp; Mission duration: <strong>{report['mission_duration_months']} months</strong>"
    html += f" &nbsp;|&nbsp; Source: <code>{report.get('csv_source', 'N/A')}</code></p>"

    # --- Completeness Check stat cards ---
    stats = [
        ("🌿 Identified", report.get("herbs_identified", 0), "#5c6bc0"),
        ("🔎 Evaluated", report.get("herbs_evaluated", 0), "#5c6bc0"),
        ("✅ Passed", report.get("herbs_passed", 0), "#4caf50"),
        ("❌ Rejected", report.get("herbs_rejected", 0), "#e57373"),
    ]
    html += "<div style='display:flex; gap:12px; justify-content:center; margin:20px 0; flex-wrap:wrap;'>"
    for label, value, color in stats:
        html += f"""
        <div style="flex:1; min-width:110px; text-align:center; background-color:#151b3d; border-top:3px solid {color}; border-radius:8px; padding:12px;">
            <div style="font-size:1.6em; font-weight:bold; color:{color};">{value}</div>
            <div style="font-size:0.85em; color:#aab4e0;">{label}</div>
        </div>"""
    html += "</div>"

    # --- Accepted countermeasures ---
    findings = report.get("findings", [])
    passed = [f for f in findings if f["status"] == "PASS"]
    html += "<h3 style='color:#81c784; border-bottom:1px solid #2e3766; padding-bottom:6px;'>✅ Accepted Countermeasures (Triangulation Mandate Satisfied)</h3>"
    if not passed:
        html += "<p style='color:#8892c2; font-style:italic;'>No herb satisfied all three legs of the Triangulation Mandate for this run.</p>"
    for f in passed:
        html += f"""
        <div style="margin-bottom:16px; background-color:#12321f; border-left:4px solid #4caf50; border-radius:6px; padding:14px 18px;">
            <h4 style="margin:0 0 6px 0; color:#a5d6a7;">🌿 {f['scientific_name']}</h4>
            <p style="margin:0 0 10px 0;"><strong>Active compounds:</strong> {f['active_compounds']}</p>
            <p style="margin:0 0 4px 0; font-size:0.9em; color:#c5cae9;"><strong>🧬 PubMed evidence:</strong></p>
            <ul style="margin:0 0 8px 0; padding-left:22px;">"""
        for e in f["pubmed_evidence"]:
            html += f"<li><a href='{e.get('url')}' target='_blank' style='color:#90caf9;'>{e.get('title', 'Untitled')}</a> — PMID {e.get('pubmed_id')} <em>({e.get('evidence_tier', 'unknown')})</em></li>"
        html += "</ul><p style='margin:0 0 4px 0; font-size:0.9em; color:#c5cae9;'><strong>🛰️ NASA GeneLab evidence:</strong></p><ul style='margin:0; padding-left:22px;'>"
        for e in f["genelab_evidence"]:
            html += f"<li><a href='{e.get('url')}' target='_blank' style='color:#90caf9;'>{e.get('title', 'Untitled')}</a> — {e.get('accession')} <em>({e.get('evidence_tier', 'unknown')})</em></li>"
        html += "</ul></div>"

    # --- Rejected herbs ---
    rejected = [f for f in findings if f["status"] == "REJECTED"]
    html += "<h3 style='color:#e57373; border-bottom:1px solid #2e3766; padding-bottom:6px;'>❌ Rejected Herbs</h3>"
    if not rejected:
        html += "<p style='color:#8892c2; font-style:italic;'>None — every identified herb passed the Triangulation Mandate.</p>"
    else:
        html += "<ul style='padding-left:22px;'>"
        for f in rejected:
            html += f"<li><strong>{f['scientific_name']}</strong> — {', '.join(f['reasons'])}</li>"
        html += "</ul>"

    # --- Limitations + footer ---
    html += "<h3 style='color:#ffb74d; border-bottom:1px solid #2e3766; padding-bottom:6px; margin-top:22px;'>⚠️ Limitations</h3><ul style='padding-left:22px; color:#c5cae9; font-size:0.92em;'>"
    for l in report.get("limitations", []):
        html += f"<li>{l}</li>"
    html += "</ul>"

    html += f"""
        <div style="text-align:center; margin-top:24px; font-size:0.85em; color:#7986cb; border-top:1px solid #2e3766; padding-top:12px;">
            <p style="margin:0 0 4px 0;">🤖 Generated by the Astro Rasayana Supervisor Agent</p>
            <p style="margin:0;">📅 {current_date}</p>
            <p style="margin-top:8px; font-style:italic;">This is a research screening tool, not a clinical or regulatory recommendation.</p>
        </div>
    </div>
    """

    display(HTML(html))


print("Display function ready: display_astro_report()")

Display function ready: display_astro_report()


## 🎛️ Step 10: Creating an Interactive Input Interface

This cell uses `ipywidgets` to build the input side of the interface:

*   **`widgets.Combobox`** for the target stressor — it doubles as a dropdown of the five `STRESSOR_PRESETS` **and** a free-text field, so a custom stressor can be typed in directly instead of needing a separate "Custom" option.
*   **`widgets.Text`** for the flora CSV path, pre-filled with the dataset resolved in Step 3.
*   **`widgets.BoundedIntText`** for an optional mission duration in months (context metadata only).
*   **Run / Clear buttons**, wired to `on_run_button_clicked` / `on_clear_button_clicked`, which call `run_astro_pipeline` and `display_astro_report` together and manage a `status_area` / `output_area` pair.

In [10]:
# Step 10: Interactive Input Interface
import ipywidgets as widgets
from ipywidgets import Layout

stressor_input = widgets.Combobox(
    value=STRESSOR_PRESETS[0],
    placeholder='Choose a preset or type a custom stressor...',
    options=STRESSOR_PRESETS,
    description='Stressor:',
    ensure_option=False,
    layout=Layout(width='100%'),
)

csv_path_input = widgets.Text(
    value=CSV_PATH,
    description='CSV path:',
    layout=Layout(width='100%'),
)

mission_duration_input = widgets.BoundedIntText(
    value=0,
    min=0,
    max=60,
    step=1,
    description='Mission (mo):',
    layout=Layout(width='250px'),
)

run_button = widgets.Button(
    description='Run Triangulation Analysis',
    button_style='success',
    tooltip='Run the Botanist -> Clinician -> Flight Surgeon -> Supervisor pipeline',
    icon='rocket',
    layout=Layout(width='260px'),
)

clear_button = widgets.Button(
    description='Clear',
    button_style='warning',
    tooltip='Clear inputs and results',
    icon='eraser',
    layout=Layout(width='100px'),
)

button_row = widgets.HBox([run_button, clear_button], layout=Layout(justify_content='center'))

status_area = widgets.Output()
output_area = widgets.Output()


def on_run_button_clicked(b):
    status_area.clear_output()
    output_area.clear_output()

    stressor = stressor_input.value.strip() or STRESSOR_PRESETS[0]
    csv_path = csv_path_input.value.strip() or CSV_PATH
    mission_duration = mission_duration_input.value or None

    with status_area:
        print(f"Running Triangulation Analysis for '{stressor}'... this may take a moment.")

    with output_area:
        report = run_astro_pipeline(csv_path, stressor, mission_duration)
        status_area.clear_output()
        display_astro_report(report)


def on_clear_button_clicked(b):
    stressor_input.value = STRESSOR_PRESETS[0]
    mission_duration_input.value = 0
    output_area.clear_output()
    status_area.clear_output()


run_button.on_click(on_run_button_clicked)
clear_button.on_click(on_clear_button_clicked)

print("Go to Step 11 & run the Final UI to use the interactive interface.")

Go to Step 11 & run the Final UI to use the interactive interface.


## 📜 Step 11: Final User Interface Integration

This final code cell assembles everything into a single interface:

*   A `widgets.Tab` separates **🛰️ Mission Parameters** (the input panel from Step 10) from **🧾 Triangulation Report** (`status_area` + `output_area`).
*   Below the tabs, one quick-launch button per entry in `STRESSOR_PRESETS` reproduces the pattern of ADA's sample-case buttons: click one, and the pipeline runs immediately for that stressor with the results shown in the report tab — no typing required, useful for quickly comparing how different stressors triangulate against the same flora dataset.
*   A closing disclaimer reiterates that this is a research screening tool.

In [11]:
# Step 11: Final User Interface Integration

def _run_preset(stressor):
    def _handler(b):
        stressor_input.value = stressor
        tabs.selected_index = 1
        on_run_button_clicked(b)
    return _handler

preset_buttons = []
for preset in STRESSOR_PRESETS:
    btn = widgets.Button(
        description=preset.title(),
        button_style='info',
        layout=Layout(width='190px', margin='4px'),
        tooltip=f"Run the pipeline for '{preset}'",
    )
    btn.on_click(_run_preset(preset))
    preset_buttons.append(btn)

input_panel = widgets.VBox([
    widgets.HTML("<h2>🛰️ Mission Parameters</h2>"),
    stressor_input,
    csv_path_input,
    mission_duration_input,
    button_row,
], layout=Layout(width='92%', margin='15px auto'))

output_panel = widgets.VBox([
    widgets.HTML("<h2>🧾 Triangulation Report</h2>"),
    status_area,
    output_area,
], layout=Layout(width='95%', margin='15px auto'))

tabs = widgets.Tab(children=[input_panel, output_panel], layout=Layout(padding='15px'))
tabs.set_title(0, '🛰️ Mission Parameters')
tabs.set_title(1, '🧾 Triangulation Report')

final_ui = widgets.VBox([
    widgets.HTML("<h1 style='text-align:center;'>🚀 Astro Rasayana ✨</h1>"),
    widgets.HTML("<div style='text-align:center; margin-bottom:16px;'>🌿 Ayurvedic Rasayana herbs, triangulated against live PubMed and NASA GeneLab evidence 🛰️</div>"),
    tabs,
    widgets.HTML("<hr style='margin:20px auto; width:80%;'>"),
    widgets.HTML("<h3 style='text-align:center;'>🧪 Quick-launch a preset stressor:</h3>"),
    widgets.HBox(preset_buttons, layout=Layout(justify_content='center', flex_flow='wrap', margin='10px 0')),
    widgets.HTML(
        "<div style='margin-top:20px; color:#8892c2; font-size:0.9em; text-align:center; border-top:1px solid #2e3766; padding-top:12px;'>"
        "ℹ️ <strong>Note:</strong> This is a research screening tool, not a clinical or regulatory recommendation.</div>"
    ),
], layout=Layout(width='95%', margin='20px auto', border='1px solid #2e3766', padding='20px', border_radius='10px'))

print("✨ Final UI Ready! ✨")
display(final_ui)

✨ Final UI Ready! ✨


This completes the implementation of Astro Rasayana as an interactive Kaggle notebook.

**Summary of Features:**

1.  **Core Engine:** Google's Gemini model driving automatic function calling across three specialist agents.
2.  **Live Evidence:** Real-time PubMed E-utilities and NASA GeneLab API calls — no cached or hallucinated citations.
3.  **Deterministic Governance:** The Triangulation Mandate PASS/REJECTED decision is rebuilt in plain Python from a logged tool-call trace, never trusted from the model's own summary.
4.  **Structured Output:** Findings assembled into a typed report dictionary shared by the display function and any downstream export.
5.  **Simplified Display:** A single `display_astro_report()` call replaces separate JSON + Markdown file writes with one rich, interactive HTML report.
6.  **Interactive UI:** An `ipywidgets`-based interface with stressor selection, CSV path, mission duration, tabs, and one-click stressor presets.

**How to Use:**

1.  Run all cells from Step 1 through Step 11 in order.
2.  In the **🛰️ Mission Parameters** tab, pick or type a target stressor, confirm the CSV path, and optionally set a mission duration.
3.  Click **Run Triangulation Analysis**.
4.  The view switches to **🧾 Triangulation Report** automatically, showing the governed PASS/REJECTED breakdown with live evidence links.
5.  Alternatively, click any of the preset buttons below the tabs to instantly triangulate that stressor.
6.  Use **Clear** to reset the inputs and results.

## ✍️ Author

This project is developed by:

*   **Dr. Debabrata Mondal** - Ayurvedic Physician
    *   GitHub: [@drdebabratamondal](https://github.com/drdebabratamondal)

This project scales clinical methodologies used for scientific validation of traditional Ayurvedic formulations into an aerospace application, pairing classical Rasayana herb classification with live PubMed and NASA GeneLab evidence.

---

## 📖 Further Exploration

*   💻 **Full source code and standalone agent scripts:** [Astro_Rasayana on GitHub](https://github.com/drdebabratamondal/astro_rasayana)
*   🧪 To run the standalone multi-agent pipeline outside this notebook: `python agents/supervisor.py`, after adding a `GEMINI_API_KEY` to a `.env` file.

---